## Exploratory Data Analysis

### Import Libraries

In [1]:
import sys
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [2]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


### Load CSV File into Polars DataFrame

In [3]:
df = pl.read_csv(
    "../couchbase_scripts/data/Financials.csv", 
    ignore_errors=True, 
    truncate_ragged_lines=True
    )

df

CUST_ID,MONTHLY_HOUSING_COST,CONTACT_PREFERENCE,CREDIT_AUTHORITY_LEVEL,CREDIT_SCORE,CREDIT_UTILIZATION,DEBT_SERVICE_COVERAGE_RATIO
str,i64,str,str,i64,f64,i64
"""CUST-417911""",3833,"""mail""","""High""",674,0.07053,1
"""CUST-758898""",3839,"""mail""","""High""",748,0.07053,1
"""CUST-958684""",6334,"""none""","""Medium""",761,0.818449,1
"""CUST-124574""",2550,"""phone""","""Very Low""",632,0.512796,1
"""CUST-198781""",2668,"""phone""","""Very Low""",706,0.512796,1
…,…,…,…,…,…,…
"""CUST-717296""",1920,"""none""","""Very Low""",816,0.003393,1
"""CUST-780694""",2014,"""none""","""Very Low""",842,0.003393,1
"""CUST-787420""",4011,"""none""","""Very High""",661,0.485421,1


### Retrieve Number of Nulls in Each Feature

In [4]:
def count_nulls(df: pl.DataFrame) -> pl.DataFrame:
    return pl.DataFrame({
        "feature": df.columns,
        "null_count": df.null_count().row(0)
    })

pl.Config.set_tbl_rows(35)

null_counts = count_nulls(df)
null_counts

feature,null_count
str,i64
"""CUST_ID""",0
"""MONTHLY_HOUSING_COST""",0
"""CONTACT_PREFERENCE""",0
"""CREDIT_AUTHORITY_LEVEL""",0
"""CREDIT_SCORE""",0
"""CREDIT_UTILIZATION""",0
"""DEBT_SERVICE_COVERAGE_RATIO""",0


### Retrieve Basic Information About DataFrame

In [5]:
def print_schema(df: pl.DataFrame):
    print(f"{'Column':<30} | {'Data Type'}")
    print("-" * 60)
    for name, dtype in zip(df.columns, df.dtypes):
        print(f"{name:<30} | {dtype}")

print_schema(df)

Column                         | Data Type
------------------------------------------------------------
CUST_ID                        | String
MONTHLY_HOUSING_COST           | Int64
CONTACT_PREFERENCE             | String
CREDIT_AUTHORITY_LEVEL         | String
CREDIT_SCORE                   | Int64
CREDIT_UTILIZATION             | Float64
DEBT_SERVICE_COVERAGE_RATIO    | Int64


### Display Summary Statistics for All Columns

In [6]:
summary = df.describe()
print(summary)

shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ CUST_ID    ┆ MONTHLY_HO ┆ CONTACT_P ┆ CREDIT_AU ┆ CREDIT_SC ┆ CREDIT_UT ┆ DEBT_SERV │
│ ---        ┆ ---        ┆ USING_COST ┆ REFERENCE ┆ THORITY_L ┆ ORE       ┆ ILIZATION ┆ ICE_COVER │
│ str        ┆ str        ┆ ---        ┆ ---       ┆ EVEL      ┆ ---       ┆ ---       ┆ AGE_RATIO │
│            ┆            ┆ f64        ┆ str       ┆ ---       ┆ f64       ┆ f64       ┆ ---       │
│            ┆            ┆            ┆           ┆ str       ┆           ┆           ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 2000       ┆ 2000.0     ┆ 2000      ┆ 2000      ┆ 2000.0    ┆ 2000.0    ┆ 2000.0    │
│ null_count ┆ 0          ┆ 0.0        ┆ 0         ┆ 0         ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ mean       ┆ null       ┆ 3279.6675  ┆ null      ┆ null      ┆ 721.6315  ┆ 

### Find Longest Text Length in Each Column

In [7]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

CUST_ID,CONTACT_PREFERENCE,CREDIT_AUTHORITY_LEVEL
u32,u32,u32
11,5,9


### Retrieve Data Types of All Columns

In [8]:
print("Column data types:\n", df.dtypes)

Column data types:
 [String, Int64, String, String, Int64, Float64, Int64]


### Count Unique Values in Each Column

In [9]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(48), f"{unique_counts}".ljust(6))

                      Unique values in CUST_ID : 1999  
         Unique values in MONTHLY_HOUSING_COST : 1684  
           Unique values in CONTACT_PREFERENCE : 4     
       Unique values in CREDIT_AUTHORITY_LEVEL : 5     
                 Unique values in CREDIT_SCORE : 249   
           Unique values in CREDIT_UTILIZATION : 1000  
  Unique values in DEBT_SERVICE_COVERAGE_RATIO : 2     


### Check Distribution of Numerical Columns

In [10]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['ID']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')

MONTHLY_HOUSING_COST
shape: (9, 2)
┌────────────┬──────────────────────┐
│ statistic  ┆ MONTHLY_HOUSING_COST │
│ ---        ┆ ---                  │
│ str        ┆ f64                  │
╞════════════╪══════════════════════╡
│ count      ┆ 2000.0               │
│ null_count ┆ 0.0                  │
│ mean       ┆ 3279.6675            │
│ std        ┆ 2002.939827          │
│ min        ┆ 650.0                │
│ 25%        ┆ 1750.0               │
│ 50%        ┆ 2704.0               │
│ 75%        ┆ 4626.0               │
│ max        ┆ 11813.0              │
└────────────┴──────────────────────┘ 


CREDIT_SCORE
shape: (9, 2)
┌────────────┬──────────────┐
│ statistic  ┆ CREDIT_SCORE │
│ ---        ┆ ---          │
│ str        ┆ f64          │
╞════════════╪══════════════╡
│ count      ┆ 2000.0       │
│ null_count ┆ 0.0          │
│ mean       ┆ 721.6315     │
│ std        ┆ 56.512083    │
│ min        ┆ 595.0        │
│ 25%        ┆ 677.0        │
│ 50%        ┆ 720.0        │
│ 75%

### List Unique Values For Certain Features

In [11]:
def list_unique_values_under_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count < threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_under_threshold(df)

Column: CUST_ID (1999 unique values)
shape: (1_999,)
Series: 'CUST_ID' [str]
[
	"CUST-100067"
	"CUST-100282"
	"CUST-100396"
	"CUST-100400"
	"CUST-100417"
	"CUST-100655"
	"CUST-100759"
	"CUST-101785"
	"CUST-101865"
	"CUST-102995"
	"CUST-103026"
	"CUST-103108"
	"CUST-103227"
	"CUST-104309"
	"CUST-104713"
	"CUST-105166"
	"CUST-106996"
	"CUST-107207"
	…
	"CUST-993400"
	"CUST-994052"
	"CUST-994211"
	"CUST-994613"
	"CUST-994795"
	"CUST-994918"
	"CUST-994965"
	"CUST-995095"
	"CUST-996204"
	"CUST-997038"
	"CUST-997072"
	"CUST-997260"
	"CUST-997681"
	"CUST-998954"
	"CUST-998988"
	"CUST-999095"
	"CUST-999182"
]
--------------------------------------------------
Column: MONTHLY_HOUSING_COST (1684 unique values)
shape: (1_684,)
Series: 'MONTHLY_HOUSING_COST' [i64]
[
	650
	651
	653
	661
	663
	669
	672
	675
	676
	682
	688
	693
	700
	701
	703
	708
	717
	718
	…
	9052
	9089
	9132
	9151
	9169
	9434
	9472
	9476
	9746
	9845
	9867
	9872
	9914
	10122
	11759
	11808
	11813
]
----------------------------------

In [12]:
def list_unique_values_over_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count > threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_over_threshold(df)

### How Many Records Remain IF I Remove Records With Any Nulls In It

In [13]:
def drop_rows_with_any_nulls(df: pl.DataFrame) -> pl.DataFrame:
    """
    Removes all rows from a Polars DataFrame that contain any null values.
    """
    return df.drop_nulls()


drop_rows_with_any_nulls(df)

CUST_ID,MONTHLY_HOUSING_COST,CONTACT_PREFERENCE,CREDIT_AUTHORITY_LEVEL,CREDIT_SCORE,CREDIT_UTILIZATION,DEBT_SERVICE_COVERAGE_RATIO
str,i64,str,str,i64,f64,i64
"""CUST-417911""",3833,"""mail""","""High""",674,0.07053,1
"""CUST-758898""",3839,"""mail""","""High""",748,0.07053,1
"""CUST-958684""",6334,"""none""","""Medium""",761,0.818449,1
"""CUST-124574""",2550,"""phone""","""Very Low""",632,0.512796,1
"""CUST-198781""",2668,"""phone""","""Very Low""",706,0.512796,1
"""CUST-228676""",5102,"""email""","""Very High""",676,0.306362,1
"""CUST-464124""",1437,"""phone""","""Low""",665,0.33702,1
"""CUST-529407""",1325,"""phone""","""Low""",708,0.33702,1
"""CUST-523002""",2323,"""email""","""High""",713,0.968387,1
